# SFINCS vs HEC-RAS comparison — Manning roughness sensitivity

Case study: **Besaya River — Corrales de Buelna**  

Compares the response of two 2D hydraulic models to the same uncertainty
in Manning roughness coefficients:

| Model | Equations | Execution |
|--------|-----------|----------|
| **SFINCS** | Simplified inertial (no advective terms) | GPU Docker / SLURM |
| **HEC-RAS 6.6** | Full 2D Saint-Venant | COM (rascontrol, Windows) |

**Note**: This notebook requires notebooks 02 and 03 to have been run first
to produce `sfincs_sensitivity_results.csv` and `hecras_sensitivity_results.csv`,
or can load data directly from TIFFs via `load_flood_ensemble`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from pyhydra.modeling.hydraulic.sensitivity import (
    load_flood_ensemble,
    build_manning_ensemble,
    flooded_area,
    spatial_stats,
    manning_flood_regression,
    filter_anomalous_simulations,
)

## Paths

In [ ]:
ZENODO_DIR       = Path("..").resolve()
SFINCS_DIR       = ZENODO_DIR / "simulations" / "SFINCS"
HECRAS_DIR       = ZENODO_DIR / "simulations" / "HEC-RAS"
COMBINATIONS_DIR = ZENODO_DIR / "generated" / "nsim_rugos"
MANNING_TIF      = ZENODO_DIR / "models" / "HEC-RAS" / "LandCover.tif"

CELL_AREA_M2 = 25.0
THRESHOLD    = 0.05

_ext_ok = ZENODO_DIR.exists()

## 1 · Load ensembles from both models

In [ ]:
if _ext_ok:
    flood_sf = load_flood_ensemble(str(SFINCS_DIR), threshold=THRESHOLD)
    flood_hr = load_flood_ensemble(str(HECRAS_DIR), threshold=THRESHOLD)

    common_sims = np.intersect1d(flood_sf.simulation.values, flood_hr.simulation.values)
    flood_sf = flood_sf.sel(simulation=common_sims)
    flood_hr = flood_hr.sel(simulation=common_sims)

    print(f"SFINCS:  {flood_sf.shape}")
    print(f"HEC-RAS: {flood_hr.shape}")
    print(f"Simulaciones comunes: {len(common_sims)}")

In [ ]:
if _ext_ok:
    sim_numbers = common_sims.tolist()

    manning = build_manning_ensemble(
        raster_path=str(MANNING_TIF),
        combinations_dir=str(COMBINATIONS_DIR),
        simulation_numbers=sim_numbers,
    )

## 2 · Per-model statistics

In [ ]:
if _ext_ok:
    stats_sf = spatial_stats(flood_sf)
    stats_hr = spatial_stats(flood_hr)
    areas_sf = flooded_area(flood_sf, CELL_AREA_M2, THRESHOLD)
    areas_hr = flooded_area(flood_hr, CELL_AREA_M2, THRESHOLD)

    summary = pd.DataFrame({
        "SFINCS_calado_medio":   stats_sf["mean"].values,
        "HECRAS_calado_medio":   stats_hr["mean"].values,
        "SFINCS_area_km2":       areas_sf * 1e-6,
        "HECRAS_area_km2":       areas_hr * 1e-6,
    }, index=common_sims)

    summary.describe().round(3)

In [ ]:
if _ext_ok:
    sf_filt_input = pd.DataFrame(
        {"depth_mean": stats_sf["mean"], "flooded_area_km2": areas_sf * 1e-6},
        index=common_sims,
    )
    hr_filt_input = pd.DataFrame(
        {"depth_mean": stats_hr["mean"], "flooded_area_km2": areas_hr * 1e-6},
        index=common_sims,
    )

    flagged, report = filter_anomalous_simulations(
        sf_filt_input, hr_filt_input,
        metrics=["depth_mean", "flooded_area_km2"],
        z_threshold=5.0,
    )

    print(f"Simulaciones anómalas detectadas: {flagged.sum()}")
    print(report.to_string())

    valid_sims  = flagged[~flagged].index.values
    flood_sf    = flood_sf.sel(simulation=valid_sims)
    flood_hr    = flood_hr.sel(simulation=valid_sims)
    manning     = manning.sel(simulation=valid_sims)
    common_sims = valid_sims

    stats_sf = spatial_stats(flood_sf)
    stats_hr = spatial_stats(flood_hr)
    areas_sf = flooded_area(flood_sf, CELL_AREA_M2, THRESHOLD)
    areas_hr = flooded_area(flood_hr, CELL_AREA_M2, THRESHOLD)

    print(f"\nSimulaciones válidas para el análisis: {len(common_sims)}")

## 2b · Detection and removal of anomalous simulations

A robust z-score (MAD-normalised) with threshold **k=5.0** is applied.

- **k=5.0** removes only evident numerical non-convergences (HEC-RAS area >1 km²,
  HEC-RAS depth >1.7 m). These are 5 simulations out of 1,000 (0.5%).
- **k=3.0** was discarded because the HEC-RAS area distribution is **bimodal**
  (two physically distinct inundation regimes); applying k=3 would eliminate both
  tails of that bimodal distribution, which is itself a scientifically relevant result.
- **IQR 1.5×** was also discarded for the same reason: it would remove ~11% of simulations.

## 3 · Compared distributions

In [ ]:
if _ext_ok:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    sns.kdeplot(stats_sf["mean"], ax=ax, label="SFINCS", fill=True, alpha=0.4)
    sns.kdeplot(stats_hr["mean"], ax=ax, label="HEC-RAS", fill=True, alpha=0.4)
    ax.set_xlabel("Calado medio espacial (m)")
    ax.set_title("Distribución del calado medio — 1.000 sims")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    sns.kdeplot(areas_sf * 1e-6, ax=ax, label="SFINCS", fill=True, alpha=0.4)
    sns.kdeplot(areas_hr * 1e-6, ax=ax, label="HEC-RAS", fill=True, alpha=0.4)
    ax.set_xlabel("Área inundada (km²)")
    ax.set_title("Distribución del área inundada — 1.000 sims")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle("SFINCS vs HEC-RAS — Besaya, T=500 años", y=1.01)
    plt.tight_layout()
    plt.show()

## 4 · Roughness ↔ depth regressions: both models

In [ ]:
if _ext_ok:
    reg_sf = manning_flood_regression(flood_sf, manning, CELL_AREA_M2, THRESHOLD)
    reg_hr = manning_flood_regression(flood_hr, manning, CELL_AREA_M2, THRESHOLD)

In [ ]:
if _ext_ok:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for ax, reg, label, color in [
        (axes[0], reg_sf, "SFINCS",  "steelblue"),
        (axes[0], reg_hr, "HEC-RAS", "darkorange"),
    ]:
        valid = reg[["manning_mean", "depth_mean"]].dropna()
        coef = np.polyfit(valid["manning_mean"], valid["depth_mean"], 1)
        x_line = np.linspace(valid["manning_mean"].min(), valid["manning_mean"].max(), 100)
        ax.scatter(valid["manning_mean"], valid["depth_mean"],
                   alpha=0.3, s=8, color=color, label=label)
        ax.plot(x_line, np.polyval(coef, x_line), color=color, lw=2,
                label=f"{label}: y={coef[0]:.2f}x+{coef[1]:.2f}")

    axes[0].set_xlabel("n Manning medio (celdas mojadas)")
    axes[0].set_ylabel("Calado medio (m)")
    axes[0].set_title("Rugosidad vs Calado")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    for ax, reg, label, color in [
        (axes[1], reg_sf, "SFINCS",  "steelblue"),
        (axes[1], reg_hr, "HEC-RAS", "darkorange"),
    ]:
        valid = reg[["manning_mean", "flooded_area_m2"]].dropna()
        coef = np.polyfit(valid["manning_mean"], valid["flooded_area_m2"] * 1e-6, 1)
        x_line = np.linspace(valid["manning_mean"].min(), valid["manning_mean"].max(), 100)
        axes[1].scatter(valid["manning_mean"], valid["flooded_area_m2"] * 1e-6,
                        alpha=0.3, s=8, color=color, label=label)
        axes[1].plot(x_line, np.polyval(coef, x_line), color=color, lw=2,
                     label=f"{label}: y={coef[0]:.2f}x+{coef[1]:.2f}")

    axes[1].set_xlabel("n Manning medio (celdas mojadas)")
    axes[1].set_ylabel("Área inundada (km²)")
    axes[1].set_title("Rugosidad vs Área inundada")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("SFINCS vs HEC-RAS — Sensibilidad a Manning", y=1.01)
    plt.tight_layout()
    plt.show()

## 5 · Inter-model coherence: depth simulation-by-simulation

In [ ]:
if _ext_ok:
    df_comp = pd.DataFrame({
        "sfincs_depth":  stats_sf["mean"],
        "hecras_depth":  stats_hr["mean"],
        "sfincs_area":   areas_sf * 1e-6,
        "hecras_area":   areas_hr * 1e-6,
    }, index=common_sims).dropna()

    r_depth, p_depth = stats.pearsonr(df_comp["sfincs_depth"], df_comp["hecras_depth"])
    r_area,  p_area  = stats.pearsonr(df_comp["sfincs_area"],  df_comp["hecras_area"])

    print(f"Correlación calado:  r={r_depth:.3f}  (p={p_depth:.2e})")
    print(f"Correlación área:    r={r_area:.3f}  (p={p_area:.2e})")

In [ ]:
if _ext_ok:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(df_comp["sfincs_depth"], df_comp["hecras_depth"], alpha=0.35, s=10)
    lim = [df_comp[["sfincs_depth","hecras_depth"]].min().min() * 0.95,
           df_comp[["sfincs_depth","hecras_depth"]].max().max() * 1.05]
    ax.plot(lim, lim, "k--", lw=1, label="1:1")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("SFINCS — calado medio (m)")
    ax.set_ylabel("HEC-RAS — calado medio (m)")
    ax.set_title(f"Calado medio sim-a-sim  (r={r_depth:.2f})")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.scatter(df_comp["sfincs_area"], df_comp["hecras_area"], alpha=0.35, s=10)
    lim2 = [df_comp[["sfincs_area","hecras_area"]].min().min() * 0.95,
            df_comp[["sfincs_area","hecras_area"]].max().max() * 1.05]
    ax.plot(lim2, lim2, "k--", lw=1, label="1:1")
    ax.set_xlim(lim2); ax.set_ylim(lim2)
    ax.set_xlabel("SFINCS — área inundada (km²)")
    ax.set_ylabel("HEC-RAS — área inundada (km²)")
    ax.set_title(f"Área inundada sim-a-sim  (r={r_area:.2f})")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle("Coherencia entre modelos — misma combinación de Manning", y=1.01)
    plt.tight_layout()
    plt.show()

## 6 · Coefficient of variation: which model is more sensitive to Manning?

In [ ]:
if _ext_ok:
    def cv(series: pd.Series) -> float:
        """Coeficiente de variación en %."""
        return series.std() / series.mean() * 100

    metrics = pd.DataFrame({
        "Métrica":  ["Calado medio (m)", "Área inundada (km²)"],
        "SFINCS_mean":  [stats_sf["mean"].mean(), (areas_sf * 1e-6).mean()],
        "SFINCS_std":   [stats_sf["mean"].std(),  (areas_sf * 1e-6).std()],
        "SFINCS_CV%":   [cv(stats_sf["mean"]),    cv(pd.Series(areas_sf * 1e-6))],
        "HECRAS_mean":  [stats_hr["mean"].mean(), (areas_hr * 1e-6).mean()],
        "HECRAS_std":   [stats_hr["mean"].std(),  (areas_hr * 1e-6).std()],
        "HECRAS_CV%":   [cv(stats_hr["mean"]),    cv(pd.Series(areas_hr * 1e-6))],
    }).set_index("Métrica")

    metrics.round(3)

## 7 · Uncertainty maps (spatial standard deviation)

In [ ]:
if _ext_ok:
    out = pd.DataFrame({
        "sfincs_depth_mean":   stats_sf["mean"],
        "sfincs_depth_median": stats_sf["median"],
        "sfincs_area_km2":     areas_sf * 1e-6,
        "sfincs_manning_mean": reg_sf["manning_mean"],
        "hecras_depth_mean":   stats_hr["mean"],
        "hecras_depth_median": stats_hr["median"],
        "hecras_area_km2":     areas_hr * 1e-6,
        "hecras_manning_mean": reg_hr["manning_mean"],
    }, index=common_sims)

    out.to_csv("comparison_clean_995.csv")
    print(f"Guardado: comparison_clean_995.csv  ({len(out)} filas)")

    report.to_csv("anomalous_simulations.csv")
    print(f"Guardado: anomalous_simulations.csv  ({len(report)} filas)")

## 8 · Save combined results

In [ ]:
if _ext_ok:
    out = pd.DataFrame({
        "sfincs_depth_mean":   stats_sf["mean"],
        "sfincs_depth_median": stats_sf["median"],
        "sfincs_area_km2":     areas_sf * 1e-6,
        "sfincs_manning_mean": reg_sf["manning_mean"],
        "hecras_depth_mean":   stats_hr["mean"],
        "hecras_depth_median": stats_hr["median"],
        "hecras_area_km2":     areas_hr * 1e-6,
        "hecras_manning_mean": reg_hr["manning_mean"],
    }, index=common_sims)

    out.to_csv("comparison_sfincs_hecras.csv")
    print(f"Guardado: comparison_sfincs_hecras.csv  ({len(out)} filas)")